### this was the PCA approach but still rendered 3D

In [2]:
### Imports ------------------------------------------------------------------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler, random_split
import time
import random
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from typing import Callable, Sequence
import copy
import plotly.graph_objects as go
from tqdm import tqdm

print(f"Cuda Check: {torch.cuda.is_available()}")

# load the data
n_samples_to_load = 5000
with h5py.File("training_data_collections/2025-01-13--10-29-19--X10000.h5", "r") as f:
    data = [{key: np.array(f[item][key]) for key in f[item]} for item in list(f.keys())[:n_samples_to_load]]

Cuda Check: True


In [3]:
### Classes ------------------------------------------------------------------------------------------------------------
class MyDataset(Dataset):
    """
    Barebone dataset class
    """
    
    def __init__(self, data: Sequence):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        return self.data[idx] 

class ScheduledBatchSampler(Sampler):
    """
    batch_sampler with batch size scheduling based on a custom dict. Use as custom batch_sampler fn in Dataloader().
    batch schedule = {how many epochs : batch size, ... : ...}. the current epoch needs to be set in the training loop by using set_epoch(E). If more epochs are trained than specified in the schedule, the last batch size is returned
    """
    
    def __init__(self, data_source: Dataset, schedule: list[tuple], shuffle: bool=False, drop_last: bool=False):
        """
        Args:
            data_source: Dataset to sample from.
            schedule   : list of tuples with (BS, nr_epochs for that BS), e.g. [(32, 2), (64, 4), (128, 8)]
            shuffle    : whether to shuffle the dataset indices.
            drop_last  : whether to drop the last incomplete batch.
        """
        
        # set up instance variables
        self._data_source   = data_source                     # this is essentially the whole dataset
        self._num_samples   = len(data_source)                # length of the whole dataset
        self._indices       = list(range(self._num_samples))  # base indices. (shuffled) batches of these are returned
        self._schedule      = self._expand_schedule(schedule) # schedule list but now with each batch size repeated * n
        self._current_epoch = None                            # to keep track of the current epoch in training loop
        self._shuffle       = shuffle
        self._drop_last     = drop_last

    def __iter__(self):
        """
        this function is essentially responsible for returning a generator object. The yield statement is kind of like a return, but instead it just advances the loop further every time the next() method is called, or in a for loop. it kind of "remebers" the last time it was called.
        """
        
        if self._current_epoch is None:
            raise ValueError(f"no epoch set! always set current epoch of sampler in training loop with set_epoch!")
        
        # determine batch size
        batch_size = self._get_batch_size()
        
        # Shuffle before returning the generator object for the new epoch
        if self._shuffle:
            random.shuffle(self._indices) 

        # determines the start indices for each batch. range(start, stop, step). 
        # Note this is essentially only the index to find the indices in self.indices for one batch xD
        for start_idx in range(0, len(self._indices), batch_size):
            # grab all the indices (now those indices corresponding to samples) belonging to a batch
            batch = self._indices[start_idx:start_idx + batch_size]
            # skip yielding the last semi-full batch if self.drop_last is true
            if len(batch) < batch_size and self._drop_last:
                continue
            # basically returns one element when it's called and then pauses the function execution until next()
            yield batch

    def __len__(self):
        """
        just returns the number of batches in the current epoch. changes by 1 depending on whether the last semi-full one is dropped or not.
        """
        
        if self._current_epoch is None:
            raise ValueError(f"no epoch set yet!")
        
        batch_size = self._get_batch_size()
        if self._drop_last:
            return math.floor(self._num_samples / batch_size)
        else:
            return math.ceil(self._num_samples / batch_size)

    def _expand_schedule(self, schedule: list[tuple]):
        """
        Expand the schedule to be a list of repeated batch size values. 
        e.g from [(64, 1), (128, 2), (256, 4)] -> [64, 128, 128, 256, 256, 256, 256]
        """
        
        expanded_schedule = []
        for batch_size, epochs in schedule:
            expanded_schedule.extend([batch_size] * epochs)
        return expanded_schedule

    def _get_batch_size(self):
        """
        just convenience for getting the currend batch size from the expanded schedule depending on current epoch. If the epoch is larger than what is specified in the schedule, it just returns the last value.
        """
        
        if self._current_epoch < len(self._schedule):
            return self._schedule[self._current_epoch]
        else:
            return self._schedule[-1]

    def get_total_epochs(self):
        """
        Returns the total number of epochs specified with the schedule
        """
        
        return len(self._schedule)
    
    def get_total_steps(self):
        """
        Returns the total number of batches that are returned by sampler. Takes drop_last into account.
        """
        
        total_steps = 0
        if self._drop_last is True:
            for bs in self._schedule:
                total_steps += math.floor(self._num_samples / bs)
        else:
            for bs in self._schedule:
                total_steps += math.ceil(self._num_samples / bs)
        return int(total_steps)
    
    def get_total_samples(self):
        """
        Returns the total number of samples that are returned by the sampler. Takes drop_last into account.
        """
        total_samples = 0
        
        if self._drop_last is True:
            for bs in self._schedule:
                total_samples += (self._num_samples // bs) * bs
        else:
            total_samples = len(self._schedule) * self._num_samples
        return int(total_samples)
    
    def set_epoch(self, epoch: int):
        """
        update the current epoch when called in the training loop
        note: could also be replaced with something like step() to just increment the epoch by one ...
        """
        
        self._current_epoch = epoch

class TestNet(nn.Module):
    def __init__(self, init_mode: str="none"):
        super().__init__()
        
        SZ = 30
        self.linear_1x1 = nn.Linear( 1, SZ)
        self.linear_1x2 = nn.Linear(SZ, SZ)
        self.linear_1x3 = nn.Linear(SZ, SZ)
        self.linear_1x4 = nn.Linear(SZ,  1)
        
        self.dropout    = nn.Dropout(p=0.00)
        self.activation = nn.Tanh()

        self._initialize_weights(mode=init_mode)
        return
        
    def _initialize_weights(self, mode: str):
        if mode == "none":
            return
        if mode not in ["xavier", "kaiming", "orthogonal"]:
            raise ValueError(f"please choose a valid method from [xavier, kaiming, orthogonal]! (got {mode})")
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if mode == "xavier":
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
                if mode == "kaiming":
                    nn.init.kaiming_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
                if mode == "orthogonal":
                    nn.init.orthogonal_(m.weight)
                    nn.init.zeros_(m.bias)
        return
    
    def forward(self, sample: dict):
        
        x = sample["input"][:, None].float()
        
        x = self.dropout(self.activation(self.linear_1x1(x)))
        x = self.dropout(self.activation(self.linear_1x2(x)))
        x = self.dropout(self.activation(self.linear_1x3(x)))
        x = self.linear_1x4(x)
        
        return x
    
    
### Functions ----------------------------------------------------------------------------------------------------------
def criterion(sample: dict, pred: torch.Tensor):
    # return torch.abs(pred - sample["label"][:, None])
    return (pred - sample["label"][:, None])**2

def do_loss(criterion: Callable, sample: dict, pred: torch.Tensor):
    loss        = criterion(sample, pred)
    loss_avg    = torch.mean(loss)
    loss_std_hi = torch.std(loss[loss > loss_avg], unbiased=False)
    loss_std_lo = torch.std(loss[loss < loss_avg], unbiased=False)
    
    return loss_avg, loss_std_hi, loss_std_lo

def do_validation(criterion: Callable, network: nn.Module, validation_dataloader: DataLoader, device: str):
    with torch.no_grad():
        loss = None
        for sample in validation_dataloader:
            sample = {key: value.to(device) for key, value in sample.items()}
            pred   = network(sample)
            
            if loss is None:
                loss = criterion(sample, pred)
            else:
                loss = torch.cat([loss, criterion(sample, pred)])
                
        loss_avg    = torch.mean(loss)
        loss_std_hi = torch.std(loss[loss > loss_avg], unbiased=False)
        loss_std_lo = torch.std(loss[loss < loss_avg], unbiased=False)
        
    return  loss_avg, loss_std_hi, loss_std_lo
    
    
### Work ---------------------------------------------------------------------------------------------------------------
DEVICE             = "cpu"

# load dataset and split
ds_full            = MyDataset(data)
ds_len             = len(ds_full)
split              = 0.81
ds_train, ds_valid = random_split(ds_full, [round(split * ds_len), ds_len - round(split * ds_len)])

# setup dataloaders and batch schedule
bs_schedule        = [(32, 8), (64, 8), (128, 8), (256, 8), (512, 8)]
sampler_train      = ScheduledBatchSampler(ds_train, bs_schedule, shuffle=True, drop_last=True)
loader_train       = DataLoader(ds_train, batch_sampler=sampler_train)
loader_valid       = DataLoader(ds_valid, batch_size=1000)
n_epochs           = sampler_train.get_total_epochs()

# initialize model, weights and optimizer
model              = TestNet(init_mode="none").to(DEVICE)
# optim              = torch.optim.SGD(model.parameters(), lr=5*(10**-4), momentum=0.9, weight_decay=0)
optim              = torch.optim.Adam(model.parameters(), lr=10**-3, betas=(0.9, 0.999), weight_decay=0)
 
trajectory = []
# Train Loop -------------------------------------------------------------------
for E in range(n_epochs):
    
    model.train()
    sampler_train.set_epoch(E)
    
    for sample in loader_train:
        
        sample = {key: value.to(DEVICE) for key, value in sample.items()}
        optim.zero_grad()
        pred =  model(sample)
        loss_avg, loss_std_hi, loss_std_lo = do_loss(criterion, sample, pred)
        loss_avg.backward()
        optim.step()
        # print(loss_avg.item())

    val_avg, val_std_hi, val_std_lo = do_validation(criterion, model, loader_valid, DEVICE)
    print(f"VALIDATION: {val_avg}")

    # save a copy of the models parameters each epoch
    trajectory.append({k: v.clone().detach().cpu() for k, v in model.state_dict().items()})

# virtually backup the model ---------------------------------------------------
model        = model.to("cpu")
STATE_BACKUP = copy.deepcopy(model.state_dict())
print("saved last model state")

VALIDATION: 1.9160542483488918
VALIDATION: 1.405430770713727
VALIDATION: 0.5208154839993048
VALIDATION: 0.2067628313637147
VALIDATION: 0.06923087901512635
VALIDATION: 0.044041721001799046
VALIDATION: 0.03275276157002214
VALIDATION: 0.027886634795308573
VALIDATION: 0.02619339099285587
VALIDATION: 0.023567556388852995
VALIDATION: 0.022458619716978313
VALIDATION: 0.021290321885442574
VALIDATION: 0.020129058760446704
VALIDATION: 0.01941961497161015
VALIDATION: 0.01929324035777685
VALIDATION: 0.01949879870103758
VALIDATION: 0.018948274519029397
VALIDATION: 0.019931558727022725
VALIDATION: 0.018029837693180572
VALIDATION: 0.018428861682520385
VALIDATION: 0.018039973984444804
VALIDATION: 0.017620262717101022
VALIDATION: 0.017768129743570563
VALIDATION: 0.01872953119869849
VALIDATION: 0.017931102554419827
VALIDATION: 0.01740530420340363
VALIDATION: 0.017431110474424173
VALIDATION: 0.017240025620745763
VALIDATION: 0.017855135338146234
VALIDATION: 0.01736153762973336
VALIDATION: 0.01705268413321

In [6]:
# generate 2 directions from PCA over parameter space
trajectory_flat = []
for params in trajectory:
    flat_params = torch.cat([p.flatten() for p in params.values()])
    trajectory_flat.append(flat_params.numpy())
    
# center the trajectory
trajectory_flat     = np.array(trajectory_flat)
trajectory_mean     = np.mean(trajectory_flat, axis=0)
trajectory_centered = trajectory_flat - trajectory_mean # [nBatches, nParams]

# perform PCA
U, S, Vt = np.linalg.svd(trajectory_centered, full_matrices=False)
principal_components = Vt[:2]
go.Figure(data=[go.Scatter(y=S, mode="markers")]).show()

# calculate param norm of state_backup
param_norm = torch.cat([p.flatten() for p in STATE_BACKUP.values()]).norm().item()
print(f"param norm at 'center point': {param_norm}")

dir1 = principal_components[0] * param_norm
dir2 = principal_components[1] * param_norm

# reshape DIRS to match the model parameters dict shape
DIR1, DIR2 = {}, {}
idx = 0
for key, value in STATE_BACKUP.items():
    shape = value.shape
    size  = value.numel()
    DIR1[key] = torch.tensor(dir1[idx:idx+size]).view(shape)
    DIR2[key] = torch.tensor(dir2[idx:idx+size]).view(shape)
    idx += size

SAMPLES = next(iter(loader_valid))
model = model.train()

param norm at 'center point': 9.024477005004883


In [7]:
# generate the loss landscape (Last Only)
N    = 16
α, β = np.array([1, 1]) * 1
a, b = torch.linspace(-1, 1, N), torch.linspace(-1, 1, N)
a, b = torch.meshgrid(a, b, indexing="ij")
Z    = torch.zeros(N, N)

EVAL_STATE = trajectory[-1]
for i in tqdm(range(N), ncols=120):
    for j in range(N):
        
        tempstate = copy.deepcopy(EVAL_STATE)
        for key, value in tempstate.items():
            tempstate[key] = tempstate[key] + α * a[i, j] * DIR1[key] + β * b[i, j] * DIR2[key]

        model.load_state_dict(tempstate)
        pred     = model(SAMPLES)
        loss     = criterion(SAMPLES, pred)
        loss_avg = torch.mean(loss)
        Z[i, j] = loss_avg.item()
        
fig = go.Figure()
fig.add_trace(go.Surface(
    x=a, y=b, z=torch.log(Z + 1e-6),
    ))

fig.update_layout(
    width=600, height=600,
    paper_bgcolor='rgba(0,0,0,1)',
    plot_bgcolor='rgba(0,0,0,0)',
    scene=dict(
        xaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray"
        ),
        yaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray"
        ),
        zaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray"
        ),
        aspectratio=dict(x=1, y=1, z=1),
    )
)

100%|███████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 31.90it/s]


In [8]:
# generate the loss landscape (Full History)
N_STATES = len(trajectory)

N    = 16
α, β = np.array([1, 1]) * 0.5
a, b = torch.linspace(-1, 1, N), torch.linspace(-1, 1, N)
a, b = torch.meshgrid(a, b, indexing="ij")
Z    = torch.zeros(N_STATES, N, N)

for n in tqdm(range(N_STATES), ncols=120):
    EVAL_STATE = trajectory[n]
    for i in range(N):
        for j in range(N):
            tempstate = copy.deepcopy(EVAL_STATE)
            for key, value in tempstate.items():
                tempstate[key] = tempstate[key] + α * a[i, j] * DIR1[key] + β * b[i, j] * DIR2[key]

            model.load_state_dict(tempstate)
            pred     = model(SAMPLES)
            loss     = criterion(SAMPLES, pred)
            loss_avg = torch.mean(loss)
            Z[n, i, j] = loss_avg.item()

# define the figure object and add the static plots and traces here (initial plots also here!)
fig = go.Figure()

# static layout constraints (initial annotations also here!)
fig.add_trace(go.Surface(
    x=a, y=b, z= torch.log(Z[0, :, :] + 1e-6),
    cmin=torch.log(Z.min() + 1e-6).item(), cmax=torch.log(Z.max() + 1e-6).item()
    ))

# create different frames (one for each time t) ------------------------------------------------------------------------
frames = []
for n in range(N_STATES):
    frames.append(go.Frame(
        # in data are all the plot / traces (like contourf, and scatter or lines)
        data = [
            go.Surface(
                x=a, y=b, z= torch.log(Z[n, :, :] + 1e-6),
                cmin=torch.log(Z.min() + 1e-6).item(), cmax=torch.log(Z.max() + 1e-6).item()
                ),
            ],
        # one name for each frame (has to match up EXACTLY with the "args" in slider!)
        name = f"state {int(n)}",
        # here are all the layout modifications (annotations or formatting)
        # layout = go.Layout()
    ))
    
# Add all the frames to the figure -------------------------------------------------------------------------------------
fig.frames = frames

# Add the slider and play button ---------------------------------------------------------------------------------------
fig.update_layout(
    sliders = [
        # this is one slider (= one dict)
        dict(
            # the active controls the initial position of the slider (just use an index)
            active = 0, 
            # here a separate dict for each timestep is defined
            steps = [
                dict(
                    method = "animate", 
                    args = [
                        # this has the be the EXACT name of a frame (= list of frame names to be shown)
                        [f"state {int(n)}"], 
                        dict(
                            mode="immediate", 
                            frame = dict(duration = 500, redraw = True)
                            )
                        ],
                    # this is just how all the ticks are labelled ... (doesnt have to match with frame name)
                    label = f"asdf {n}"
                ) for n in range(N_STATES)
            ],
            transition = dict(duration = 0),
            x = 0.1,
            xanchor = "left",
            y = 0,
            yanchor = "top",
            currentvalue = dict(prefix = "Step: ")
        )
    ],
    uirevision = "const",
    width=600, height=600,
    paper_bgcolor='rgba(0,0,0,1)',
    plot_bgcolor='rgba(0,0,0,0)',
    scene=dict(
        xaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray"
        ),
        yaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray"
        ),
        zaxis=dict(
            backgroundcolor="black",
            color="white",
            gridcolor="gray",
            range=[torch.log(Z.min() + 1e-6), torch.log(Z.max() + 1e-6)]
        ),
        aspectratio=dict(x=1, y=1, z=1)
    ),
)    
 
fig.show()           

100%|███████████████████████████████████████████████████████████████████████████████████| 40/40 [00:19<00:00,  2.02it/s]
